# 🎬 VideoClipper — Colab Deploy

### Before running:
1. In Google Drive → **New → Folder upload** → select your `Web_clipper` folder
2. Wait for upload to finish
3. Get a free ngrok auth token from [ngrok.com/signup](https://ngrok.com/signup) → Dashboard → Your Authtoken
4. Paste the token in **Cell 6**
5. **Runtime → Run all**

---
**Clips are saved to Google Drive** → persist across sessions.

In [ ]:
# ── Cell 1: Mount Google Drive ─────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
# ── Cell 2: Copy project from Drive ───────────────────────
import os, shutil

# Adjust this if your folder has a different name in Drive
DRIVE_SRC  = '/content/drive/MyDrive/Web_clipper'
DEPLOY_DIR = '/content/webclipper'

if not os.path.exists(DRIVE_SRC):
    # Try common name variants
    for candidate in ['Web_clipper', 'web_clipper', 'WebClipper', 'webclipper']:
        p = f'/content/drive/MyDrive/{candidate}'
        if os.path.exists(p):
            DRIVE_SRC = p
            break
    else:
        raise FileNotFoundError(
            'Could not find project folder in Drive. '
            f'Looked for: {DRIVE_SRC}\n'
            'Check the exact folder name in your Drive and update DRIVE_SRC above.'
        )

if os.path.exists(DEPLOY_DIR):
    shutil.rmtree(DEPLOY_DIR)

shutil.copytree(DRIVE_SRC, DEPLOY_DIR)
print('Project copied from Drive to', DEPLOY_DIR)
print('Files found:', os.listdir(DEPLOY_DIR))

In [ ]:
# ── Cell 3: Install system packages ───────────────────────
import subprocess

print('Installing FFmpeg + Chromium dependencies...')
subprocess.run([
    'apt-get', 'install', '-y', '-q',
    'ffmpeg',
    'libnss3', 'libatk-bridge2.0-0', 'libdrm2',
    'libxkbcommon0', 'libgbm1', 'libasound2',
    'libxcomposite1', 'libxdamage1', 'libxrandr2',
    'libpangocairo-1.0-0', 'libcairo-gobject2', 'libgtk-3-0'
], check=True)

result = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True)
print('FFmpeg OK:', result.stdout.split('\n')[0])

In [ ]:
# ── Cell 4: Install Python packages ───────────────────────
import subprocess

print('Installing Python packages...')
r = subprocess.run(
    ['pip', 'install', '--upgrade', 'flask', 'flask-cors', 'yt-dlp', 'playwright', 'pyngrok'],
    capture_output=True, text=True
)
print(r.stdout[-3000:] if r.stdout else '')
if r.returncode != 0:
    print('pip stderr:', r.stderr[-2000:])
    raise RuntimeError('pip install failed — see output above')

print('Installing Playwright Chromium...')
subprocess.run(['playwright', 'install', 'chromium'], check=True)
subprocess.run(['playwright', 'install-deps', 'chromium'], check=True)
print('All packages installed.')

In [ ]:
# ── Cell 5: Persistent clip storage on Google Drive ───────
import os

DRIVE_TMP = '/content/drive/MyDrive/webclipper_tmp'
APP_TMP   = '/content/webclipper/tmp'

for sub in ['clips', 'raw', 'cookies']:
    os.makedirs(f'{DRIVE_TMP}/{sub}', exist_ok=True)

if os.path.exists(APP_TMP) and not os.path.islink(APP_TMP):
    import shutil
    shutil.rmtree(APP_TMP)
if not os.path.exists(APP_TMP):
    os.symlink(DRIVE_TMP, APP_TMP)

print('Clips will be saved to Google Drive →', DRIVE_TMP)

In [ ]:
# ── Cell 6: Set ngrok auth token ──────────────────────────
# Free token from: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = 'PASTE_YOUR_TOKEN_HERE'

from pyngrok import ngrok, conf
conf.get_default().auth_token = NGROK_TOKEN
print('ngrok token set.')

In [ ]:
# ── Cell 7: Start VideoClipper ────────────────────────────
import subprocess, threading, time, os
from pyngrok import ngrok

os.chdir('/content/webclipper/backend')

flask_process = [None]

def run_flask():
    env = os.environ.copy()
    env['PATH'] = env['PATH'] + ':/usr/bin'
    flask_process[0] = subprocess.Popen(
        ['python', 'app.py'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=env
    )
    for line in flask_process[0].stdout:
        print('[Flask]', line.decode().rstrip())

threading.Thread(target=run_flask, daemon=True).start()
time.sleep(4)

# Kill any local ngrok processes
subprocess.run(['pkill', '-f', 'ngrok'], capture_output=True)
time.sleep(2)
ngrok.kill()
time.sleep(2)

# Retry loop — ngrok cloud can take time to release old endpoint
tunnel = None
for attempt in range(6):
    try:
        tunnel = ngrok.connect(5000)
        break
    except Exception as e:
        err = str(e)
        if 'ERR_NGROK_334' in err or 'already online' in err:
            print(f'Old tunnel still online, waiting... ({attempt+1}/6)')
            time.sleep(10)
            ngrok.kill()
            time.sleep(2)
        else:
            raise

if tunnel is None:
    raise RuntimeError(
        'Could not start ngrok tunnel after 6 attempts.\n'
        'Go to https://dashboard.ngrok.com/tunnels/agents and disconnect the old agent, then re-run this cell.'
    )

public_url = tunnel.public_url

print()
print('=' * 60)
print('  VideoClipper is LIVE!')
print(f'  URL: {public_url}')
print('=' * 60)
print('Open on phone, laptop, any device.')
print('Keep this tab open — closing Colab stops the app.')

In [ ]:
# ── Optional: Stop the server ─────────────────────────────
ngrok.kill()
if flask_process[0]:
    flask_process[0].terminate()
print('Server stopped.')